# Single-Stage Toy Objective

## What you will learn
How coupled plasma and coil terms can be optimized together using continuation.

## Codes used
Pure Python toy objective; optional JAX/SIMSOPT research extension.

## Run mode
This notebook uses RUN_MODE = "cached" by default. Allowed values are "tiny", "cached", and "research".

## Expected outputs
`05_single_stage_history.png` and `05_weight_continuation.png`.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "sos2026").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Keep RUN_MODE='cached' first; install requirements-colab.txt from the cloned repo if needed.")
else:
    print("Local runtime detected.")

In [ ]:
RUN_MODE = "cached"  # allowed: "tiny", "cached", "research"
print(f"RUN_MODE = {RUN_MODE}")

In [ ]:
import importlib
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sos2026.paths import PROJECT_ROOT, DATA_DIR, CACHE_DIR, FIGURE_DIR, MOVIE_DIR, ensure_directories
ensure_directories()
print("Figures:", FIGURE_DIR)
print("Cached data:", CACHE_DIR)

## 1. Learning frame

This notebook is a deliberately small project: define one metric, produce one plot, expose one failure mode, and identify where a real code would enter.

In [ ]:
from sos2026.plotting import savefig, caption

## 2. Load or generate the teaching data

Cached mode uses small arrays so the conceptual workflow is always available.

In [ ]:
steps = np.arange(50)
J_plasma = 0.7*np.exp(-steps/10) + 0.05
J_flux = 1.0*np.exp(-steps/15) + 0.08
J_coil = 0.55*np.exp(-steps/25) + 0.12
history = pd.DataFrame({"step": steps, "J_plasma": J_plasma, "J_flux": J_flux, "J_coil": J_coil})
history.head()

## 3. Make the primary plot

Every plot has a one-sentence caption because students should know how to read it without guessing.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.8))
for col in ["J_plasma", "J_flux", "J_coil"]:
    ax.plot(history["step"], history[col], lw=2, label=col)
ax.set_xlabel("continuation step")
ax.set_ylabel("objective contribution")
ax.set_title("Single-stage objective history")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
caption(ax, "Plotting objective pieces prevents a scalar total from hiding an unbuildable optimum.")
savefig(fig, "05_single_stage_history.png")
plt.show()

## 4. Probe the metric

A metric becomes useful for optimization only when we understand how it changes across design choices.

In [ ]:
w = np.logspace(-2, 1.2, 60)
flux = 0.10 + 0.9/(1 + 3*w)
complexity = 0.8 + 0.22*np.log10(1 + 10*w)
fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.plot(w, flux, label="flux mismatch")
ax.plot(w, complexity, label="coil complexity")
ax.set_xscale("log")
ax.set_xlabel("flux penalty weight")
ax.set_ylabel("normalized metric")
ax.set_title("Weight continuation")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
caption(ax, "Weights are design choices; continuation makes their effects visible.")
savefig(fig, "05_weight_continuation.png")
plt.show()

## 5. Interpret the design consequence

The table below translates the plot into an optimization decision.

In [ ]:
terms = pd.DataFrame({
    "term": ["J_plasma", "J_flux", "J_coil", "J_separation", "J_curvature"],
    "protects": ["magnetic quality", "target surface", "buildability", "coil clearance", "manufacturing"],
    "failure_if_missing": ["poor confinement", "coils do not make the plasma", "wild coils", "collisions/access loss", "unbuildable bends"],
})
terms

## 6. Failure mode

The cached plot is useful only if we say what it does not prove.

In [ ]:
failure_mode = pd.DataFrame({
    "cached_mode_proves": ["workflow shape", "plot grammar", "where the metric enters"],
    "cached_mode_does_not_prove": ["validated physics", "final design ranking", "runtime scalability"],
})
failure_mode

## 7. Research-mode hook

Run this cell only after timing the package on the lecture machine.

In [ ]:
if RUN_MODE == "research":
    print("Research path: replace toy terms with differentiable VMEC-JAX/SIMSOPT quantities.")
else:
    print("Cached mode: research package path skipped intentionally.")

## 8. Mini project handoff

Use this notebook during the lecture as the computational project slide points to: change one parameter, regenerate one plot, and explain one design tradeoff.

In [ ]:
project_steps = pd.DataFrame({
    "step": [1, 2, 3, 4],
    "action": ["identify metric", "change one input", "regenerate plot", "state failure mode"],
})
project_steps

## Try this
Change one scalar or one row in the cached data and regenerate the primary plot.

## Expected qualitative answer
The plot should move in a physically interpretable direction, but the cached result remains an educational proxy.

## Research extension
Replace the cached data source with the corresponding real package output after timing and API verification.